# 03 – Forecasting Models
Ziel: Alle 4 Modelle trainieren und Forecast-Metriken vergleichen.

In [ ]:
import importlib
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

import src.utils.metrics as metrics_module
import src.forecasting.train as train_module
import src.forecasting.evaluate_forecast as evaluate_module
import src.utils.plotting as plotting_module
from src.models import (linear_regression, random_forest, neural_net,
                        quantile_regression, persistence_model,
                        quantile_regression_forest, arima, decomposition)

importlib.reload(metrics_module)
importlib.reload(train_module)
importlib.reload(evaluate_module)
importlib.reload(plotting_module)

prepare_data = train_module.prepare_data
scale_features = train_module.scale_features
evaluate_all = evaluate_module.evaluate_all
plot_forecast_vs_actual = plotting_module.plot_forecast_vs_actual
plot_metric_comparison = plotting_module.plot_metric_comparison
from src.utils.plotting import plot_quantile_forecast
print(f'Plots werden gespeichert unter: {plotting_module.SAVE_PATH}')

## Daten laden

In [ ]:
df = pd.read_csv('../data/processed/final_dataset.csv', parse_dates=['timestamp'])
print(df.shape)
df.head()

## Train/Test Split
`shuffle=False` ist bei Zeitreihen Pflicht – keine zufällige Aufteilung!

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(df)
X_sc_train, X_sc_test, scaler = scale_features(X_train, X_test)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Features: {X_train.shape[1]}')

## 0. Persistence Model
: ŷ(t) = y(t−1) — letzter Zeitschritt  
: ŷ(t) = y(t−24) — gleiche Stunde Vortag

In [ ]:
# 1h-Persistence: ŷ(t) = y(t-1)
pm_model    = persistence_model.train(X_train, y_train, lag=1)
y_pred_pm   = persistence_model.predict(pm_model, X_test, y_test)
plot_forecast_vs_actual(y_test, y_pred_pm, "Persistence (1h)")

# 24h-Persistence: ŷ(t) = y(t-24) – gleiche Stunde Vortag
pm24_model  = persistence_model.train(X_train, y_train, lag=24)
y_pred_pm24 = persistence_model.predict(pm24_model, X_test, y_test)
plot_forecast_vs_actual(y_test, y_pred_pm24, "Persistence (24h)")

## 1. Linear Regression

In [ ]:
lr_model  = linear_regression.train(X_train, y_train, polynomial_degree=2)
y_pred_lr = linear_regression.predict(lr_model, X_test)
plot_forecast_vs_actual(y_test, y_pred_lr, 'Linear Regression')

## 2. Random Forest

In [ ]:
rf_model  = random_forest.train(X_train, y_train)
y_pred_rf = random_forest.predict(rf_model, X_test)
plot_forecast_vs_actual(y_test, y_pred_rf, 'Random Forest')

## 3. Neural Network

In [ ]:
nn_model  = neural_net.train(X_sc_train, y_train, epochs=100)
y_pred_nn = neural_net.predict(nn_model, X_sc_test)
plot_forecast_vs_actual(y_test, y_pred_nn, 'Neural Network')

## 4. Quantile Regression

In [ ]:
qr_models = quantile_regression.train_all_quantiles(X_train, y_train)
y_pred_q50 = quantile_regression.predict_quantile(qr_models, X_test, 0.5)

from src.utils.plotting import plot_quantile_forecast
q10 = quantile_regression.predict_quantile(qr_models, X_test, 0.1)
q90 = quantile_regression.predict_quantile(qr_models, X_test, 0.9)
plot_quantile_forecast(y_test, q10, y_pred_q50, q90)

## 5. Quantile Regression Forest

In [ ]:
qrf_model  = quantile_regression_forest.train(X_train, y_train)
y_pred_qrf = quantile_regression_forest.predict(qrf_model, X_test, quantile=0.5)

qrf_q10 = quantile_regression_forest.predict(qrf_model, X_test, quantile=0.1)
qrf_q90 = quantile_regression_forest.predict(qrf_model, X_test, quantile=0.9)
plot_quantile_forecast(y_test, qrf_q10, y_pred_qrf, qrf_q90)

## 6. ARIMA

In [ ]:
arima_model  = arima.train(y_train)
y_pred_arima = arima.predict(arima_model, steps=len(y_test))
plot_forecast_vs_actual(y_test, y_pred_arima, "ARIMA")

## 7. Zeitreihen-Dekomposition
Die Zeitreihe wird in **Trend**, **Saisonalität** und **Residuum** zerlegt.
Der Forecast kombiniert einen extrapolierten linearen Trend mit dem
wiederholten Saisonmuster (Periode = 24 h).

In [ ]:
decomp_model   = decomposition.train(y_train)
y_pred_decomp  = decomposition.predict(decomp_model, steps=len(y_test))

# Dekompositions-Plot (Trend, Saisonalität, Residuum)
import matplotlib.pyplot as plt
dec = decomp_model["decomposition"]
fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
axes[0].plot(dec.observed,   label="Beobachtet",  linewidth=0.6)
axes[0].set_ylabel("Beobachtet")
axes[1].plot(dec.trend,      label="Trend",       linewidth=1.2, color="tab:orange")
axes[1].set_ylabel("Trend")
axes[2].plot(dec.seasonal,   label="Saisonalität",linewidth=0.8, color="tab:green")
axes[2].set_ylabel("Saisonalität")
axes[3].plot(dec.resid,      label="Residuum",    linewidth=0.6, color="tab:red")
axes[3].set_ylabel("Residuum")
for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
fig.suptitle("Additive Zeitreihen-Dekomposition (Trainingsdaten)", fontsize=13)
plt.tight_layout()
plt.savefig(plotting_module.SAVE_PATH / "decomposition_components.png", dpi=150, bbox_inches="tight")
plt.show()

plot_forecast_vs_actual(y_test, y_pred_decomp, "Dekomposition")

## Vergleich: Forecast-Metriken
**Wichtig**: Das ist NICHT die finale Bewertung – die kommt in Notebook 04.

In [ ]:
models = {
    'Persistence':        pm_model,
    'Persistence24h':     pm24_model,
    'LinearRegression':   lr_model,
    'RandomForest':       rf_model,
    'NeuralNet':          nn_model,
    'QuantileRegression': qr_models,
    'QRF':                qrf_model,
    'ARIMA':              arima_model,
    'Decomposition':      decomp_model,
}
df_results = evaluate_all(models, X_test, X_sc_test, y_test)
df_results

In [ ]:
plot_metric_comparison(df_results.to_dict('index'), metric='rmse')

## Key Insight
Das RMSE-Ranking sagt noch nichts über den wirtschaftlichen Wert aus.
→ Weiter zu Notebook 04 für die echte Evaluation.